# KV-cache pruning experiments on Kaggle (2x T4)

**One-time setup:** phone-verify your Kaggle account. In this notebook's right panel set **Accelerator = GPU T4 x2** and **Internet = On**.

**How to run a stage unattended (recommended):**
1. Set `STAGE` in the next cell.
2. Click **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours; you can close the browser.
3. When it finishes, open the version's **Output** tab and download `results.zip` (also commit the `results/` files to GitHub).

**Next stage:** click **Add Input → Your Work → this notebook's previous version**, change `STAGE`, and Save Version again. Earlier results are restored automatically and finished rows are skipped.

| STAGE | What runs | Approx. time on 2x T4 |
| --- | --- | --- |
| `main_20` | static profile calibration + main experiment at 20% budget | 4-5 h |
| `main_rest` | main experiment at 10%, 30%, 50% | ~10 h (stops safely at the limit; rerun to finish) |
| `passkey_ablation` | passkey heatmaps (GPU 0) + tau/floor/random ablations (GPU 1) | 4-6 h |
| `window_profile` | window 16 / 64 ablations + memory/latency profiling + passkey at 2%/5% | ~2 h |
| `confirm` | passkey at 2%/3% of 4K: entropy vs reversed entropy vs SnapKV vs CAKE-style | ~30 min |
| `xplat` | cross-platform comparison on the T4 (Qwen2.5-0.5B, same settings as Mac and CPU) | ~30 min |
| `llama` | Llama-3.2-3B at 10% and 20% + passkey direction test (needs HF_TOKEN secret) | ~4 h |
| `figures` | only regenerate figures from restored results | 2 min |

Every stage stops cleanly before Kaggle's 12-hour limit. If a stage prints `time limit ... reached`, run the **same** STAGE again with the previous version attached as input; it continues where it stopped.

In [ ]:
STAGE = "main_rest"   # main_20 | main_rest | passkey_ablation | window_profile | confirm | llama | xplat | figures
REPO = "https://github.com/khushpatel3016/kv-cache-entropy-pruning.git"
import time; T_START = time.time()

In [ ]:
import os, subprocess, glob, shutil, time, sys
os.chdir("/kaggle/working")
if not os.path.exists("kvp"):
    subprocess.run(["git", "clone", "-q", REPO, "kvp"], check=True)
os.chdir("/kaggle/working/kvp")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.1", "datasets==3.6.0",
                "rapidfuzz", "tabulate", "pyyaml", "-e", "."], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
os.makedirs("results", exist_ok=True); os.makedirs("profiles", exist_ok=True)

# Download and unpack LongBench once, before parallel jobs start (avoids two processes extracting at once).
subprocess.run([sys.executable, "-c", "from kvprune.data.longbench import load_longbench; print(len(load_longbench('qasper')), 'qasper samples')"],
               env={**os.environ, "PYTHONPATH": "src"}, check=True)

# Restore results and profiles from any attached input (earlier versions of this notebook or a dataset).
for f in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):
    dst = os.path.join("results", os.path.basename(f))
    if not os.path.exists(dst) or os.path.getsize(f) > os.path.getsize(dst):
        shutil.copy(f, dst); print("restored", f)
for f in glob.glob("/kaggle/input/**/profiles/*.json", recursive=True):
    shutil.copy(f, "profiles/"); print("restored", f)

In [ ]:
# Hugging Face token (only needed for Llama-3.2). Add it under Add-ons > Secrets as HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("no HF token (fine for Qwen):", type(e).__name__)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
# All stage logic lives in the repo (scripts/kaggle_stage.py), so fixes arrive with `git push`.
# The stage stops cleanly before Kaggle's 12-hour limit; rerun the same STAGE to continue.
if STAGE != "figures":
    hours_left = 11.6 - (time.time() - T_START) / 3600 - 0.2   # keep ~12 min for figures + saving
    subprocess.run([sys.executable, "scripts/kaggle_stage.py", "--stage", STAGE, "--hours", f"{hours_left:.2f}"],
                   env={**os.environ, "PYTHONPATH": "src"})

In [ ]:
subprocess.run([sys.executable, "scripts/make_figures.py", "--results", "results/*.jsonl", "--out", "figures"])
subprocess.run(["zip", "-qr", "/kaggle/working/results.zip", "results", "profiles", "figures"])
for f in sorted(glob.glob("results/*.jsonl")):
    print(f, sum(1 for _ in open(f)), "rows")
print("Download results.zip from the Output tab.")